# 01. Load & Clean

매출 CSV를 읽어 날짜·결측치·이상치를 점검하고 `sales_clean.csv`로 저장합니다.

샘플 데이터: `../data/sample_sales.csv` (24개월 × 3 상품)

본인 데이터: `INPUT_PATH`를 `../data/sales.csv`로 바꾸세요.

In [ ]:
!pip install -r ../../../requirements.txt

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
INPUT_PATH = DATA_DIR / 'sample_sales.csv'   # ← 본인 데이터로 바꿀 때 'sales.csv'
OUTPUT_PATH = DATA_DIR / 'sales_clean.csv'

df = pd.read_csv(INPUT_PATH, parse_dates=['date'])
print(f'{len(df)} rows, {df["product_id"].nunique()} products, '
      f'{df["date"].min().date()} ~ {df["date"].max().date()}')
df.head()

72 rows, 3 products, 2024-04-01 ~ 2026-03-01


,date,product_id,product_name,category,units_sold,revenue,channel
0,2024-04-01,SKU-A001,라이트 셔츠 화이트 M,셔츠,193,8685000,온라인
1,2024-04-01,SKU-B002,겨울 코트 블랙 L,코트,115,32200000,오프라인
2,2024-04-01,SKU-C003,기본 양말 3족 세트,잡화,347,3123000,온라인
3,2024-05-01,SKU-A001,라이트 셔츠 화이트 M,셔츠,202,9090000,온라인
4,2024-05-01,SKU-B002,겨울 코트 블랙 L,코트,70,19600000,오프라인


## 결측치 점검

In [3]:
missing = df.isna().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'None')

Missing values per column:
None


## 이상치 점검 (3-시그마)

각 `product_id`별로 평균 ± 3σ를 벗어나는 행에 **`outlier` 컬럼만 추가**합니다.

- 원본 `df`는 그대로 (행 수도 컬럼 수도 보존, `outlier` 한 컬럼만 추가)
- 자동 제거하지 않음 — 학생이 직접 검토해 결정
- 3-σ 기준은 보수적이므로 **0건이 자주 정상**입니다 (sample 데이터에서도 보통 0건)


In [4]:
def flag_outliers_per_group(s: 'pd.Series', k: float = 3.0) -> 'pd.Series':
    """Series 단위 z-score 기반 outlier flag. groupby().transform()용."""
    mu = s.mean()
    sd = s.std() or 1.0
    return (s - mu).abs() > k * sd


# df 재할당 없이 outlier 컬럼만 추가 (groupby + transform 패턴)
df['outlier'] = df.groupby('product_id')['units_sold'].transform(flag_outliers_per_group)

n_out = int(df['outlier'].sum())
print(f'Total rows preserved: {len(df)}  (3 products × 24 months = 72 expected)')
print(f'Outliers flagged: {n_out} rows  ({n_out / len(df) * 100:.1f}%)')

# 이상치만 별도 view (df는 그대로 유지)
outliers_view = df.loc[df['outlier'], ['date', 'product_id', 'units_sold']]
if outliers_view.empty:
    print('\n→ 이상치 없음 — sample 데이터에서는 정상 (3σ는 보수적 기준)')
else:
    print(f'\n--- {len(outliers_view)} outlier rows ---')
outliers_view.head()


Total rows preserved: 72  (3 products × 24 months = 72 expected)
Outliers flagged: 0 rows  (0.0%)

→ 이상치 없음 — sample 데이터에서는 정상 (3σ는 보수적 기준)


,date,product_id,units_sold


## 날짜 누락 점검

월별 데이터를 가정 — 상품마다 모든 월이 있는지.

In [5]:
all_months = pd.date_range(df['date'].min(), df['date'].max(), freq='MS')
for pid, g in df.groupby('product_id'):
    missing_dates = set(all_months) - set(g['date'])
    if missing_dates:
        print(f'{pid}: missing {len(missing_dates)} months')
    else:
        print(f'{pid}: complete ({len(g)} rows)')

SKU-A001: complete (24 rows)
SKU-B002: complete (24 rows)
SKU-C003: complete (24 rows)


## 저장

In [6]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved → {OUTPUT_PATH.resolve()}')

Saved → /Users/sungjae-cha/Documents/06 아름다운서당/next-seodang/projects/03_sales_inventory_tracker/data/sales_clean.csv


## 다음 단계

- `02_analyze_trends.ipynb` 시각화
- 결측·이상치가 많으면 Claude Desktop의 `data_diagnosis(si)` 프롬프트로 진단 권장